# What We Collected & How to Use It

This notebook covers two things:

1. **Projects in this dataset** — what ORACC projects we collected, their catalogues, and how to load them
2. **Reference datasets bundled with the package** — provenance mappings, historical periods, sign list, POS tags, language codes, and state/region supergroups

| Data | Location | How to access |
|---|---|---|
| **Project catalogues** (raw ORACC metadata, 1 row/text) | `enriched_data/catalogues/` | `load_project_catalogue()` |
| **Word CSVs** (parsed text content, 1 row/word) | `enriched_data/oracc_csvs/` | `load_word_csvs_from_dir()` (downloaded on demand) |
| **Provenance (cities → Pleiades)** | Bundled CSV | `reference_data.get_provenance()` |
| **Historical periods → year ranges** | Bundled CSV | `reference_data.get_period_mapping()` |
| **Cuneiform sign readings** | Bundled CSV | `reference_data.get_sign_list()` |
| **POS tag meanings** | Bundled CSV | `reference_data.get_pos_tags()` |
| **Language codes** | Bundled CSV | `reference_data.get_languages()` |
| **State/region supergroups** | Bundled CSV | `reference_data.get_state_supergroup_mapping()` |

In [1]:
# Install oracc-parser from PyPI
!pip install oracc-parser -q

# --- Optional: persist downloaded data across Colab sessions using Google Drive ---
# Without this, data downloads to /content/oracc_data and is lost when the session ends.
# Uncomment the lines below to mount your Drive and store data there persistently.
#
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["ORACC_DATA_DIR"] = "/content/drive/MyDrive/oracc_data"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 5.5 MB/s eta 0:00:00
Mounted at /content/drive


In [4]:
from oracc_parser import reference_data, load_project_catalogue
from oracc_parser.io.word_csv import load_word_csvs_from_dir
from oracc_parser.settings import WORD_CSV_DIR, CATALOGUE_DIR
import pandas as pd

## 1. Projects in This Dataset

The Zenodo record (doi: 10.5281/zenodo.20625379) contains pre-processed data for 139 ORACC projects — a subset of the full ORACC corpus. For each project we provide:

- A **catalogue CSV** (`enriched_data/catalogues/{project_slug}.csv`) with one row per text and all raw ORACC metadata fields preserved
- A folder of **word CSVs** (`enriched_data/oracc_csvs/{project_slug}/`) with one CSV per text and one row per word

Word CSVs are stored in 32 umbrella zip files on Zenodo (one per corpus group, e.g. `saao.zip` covers all `saao/*` projects) and are downloaded automatically the first time you access a project. The summary below is built from catalogue files currently on disk.

> For the live count from ORACC's servers, use `reference_data.get_live_project_list()`.

### ORACC catalogues

The ORACC catalogues contain metadata on the cuneiform documents per project.

> ⚠️ Note: catalogue entries across ORACC projects are not consistent. While some columns appear in all projects (e.g. `langs`, `project`, etc.), others are unique for some projects or even individual projects.

The raw catalogues can be viewed through `load_project_catalogue` (see next cell).

This package simplified most of the metadata fields which can also be accessed and it is the version of the metadata that is stored locally in flat CSVs with the processed documents (see next section).

In [5]:
# Load and inspect a specific project's catalogue
PROJECT = "babcity"
slug = PROJECT.replace("/", "-")

catalogue = load_project_catalogue(CATALOGUE_DIR / f"{slug}.csv")
print(f"Catalogue for {PROJECT}: {len(catalogue)} texts")
print(f"Fields: {list(catalogue.columns)}")
display(catalogue.head())

Catalogue for babcity: 276 texts
Fields: ['langs', 'project', 'credits', 'designation', 'genre', 'id_text', 'language', 'material', 'museum_no', 'object_type', 'period', 'primary_publication', 'provenience', 'public', 'images', 'text_id', 'date_remarks', 'archive', 'pleiades_id', 'pleiades_coord', 'publication_history', 'subgenre', 'trans', 'accession_no', 'buy_book']


,langs,project,credits,designation,genre,id_text,language,material,museum_no,object_type,...,text_id,date_remarks,archive,pleiades_id,pleiades_coord,publication_history,subgenre,trans,accession_no,buy_book
0,0x08000000,babcity,Lindsey Post,BE 8/1 003,Legal,P257583,Akkadian,clay,CBS 00018,tablet,...,P257583,,,,,,,,,
1,0x08000000,babcity,Lindsey Post,BE 8/1 007,Legal,P259107,Akkadian,clay,CBS 01788,tablet,...,P259107,618 BC,,,,,,,,
2,0x08000000,babcity,Based on a transliteration by Heather D. Baker...,"BE 10, 056",Legal,P261352,Akkadian,clay,CBS 05160,tablet,...,P261352,423-404 BC,Nippur 7.10.2.4 (Murašû),912910,"[45.230832,32.126944]",Clay Aram.Endors. 306 #17(Aram.only); PBS 02/1...,Receipt for house rent,['en'],,
3,0x08000000,babcity,,"BE 10, 003",,P261466,Akkadian,clay,CBS 05272,tablet,...,P261466,,,912910,"[45.230832,32.126944]",,,,,
4,0x08000000,babcity,,"BE 10, 002",,P261471,Akkadian,clay,CBS 05277,tablet,...,P261471,,,912910,"[45.230832,32.126944]",,,,,


## 2. ORACC Project Catalogue (Bundled Metadata)

A static snapshot of metadata for **all known ORACC projects** — names, languages, URLs, and whether a project has parseable text content. This is bundled with the package and loads without any download under `reference_data`.

`get_projects_metadata()` was created by the package to store metadata about the languages of the texts in the project, and whether the project is an umbrella project or not — umbrella project usually have empty text folders since the texts themselves are stored in the subprojects.

In [6]:
projects = reference_data.get_projects_metadata()
print(f"🌍 {len(projects)} ORACC projects in the bundled catalogue")
display(projects.head(10))

🌍 222 ORACC projects in the bundled catalogue


,Project,Link,Project_Name,Languages,Is_Umbrella_Project,Is_Text_Folder_Empty,Last_Check
0,adsd: adsd/Astronomical Diaries and Related Texts,http://oracc.museum.upenn.edu/adsd,adsd,Akkadian,yes,yes,14/05/2025
1,adsd/adart1: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart1,adsd/adart1,Akkadian,no,no,14/05/2025
2,adsd/adart2: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart2,adsd/adart2,Akkadian,no,no,14/05/2025
3,adsd/adart3: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart3,adsd/adart3,Akkadian,no,no,14/05/2025
4,adsd/adart5: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart5,adsd/adart5,Akkadian,no,no,14/05/2025
5,adsd/adart6: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart6,adsd/adart6,Akkadian,no,no,14/05/2025
6,AEMW: Akkadian in the Eastern Mediterranean World,https://oracc.museum.upenn.edu/aemw,aemw,"Akkadian, Ugaritic",yes,no,14/05/2025
7,aemw/idrimi: Statue of Idrimi,http://oracc.museum.upenn.edu/aemw/alalakh/idrimi,aemw/alalakh/idrimi,Akkadian,no,no,14/05/2025
8,aemw/amarna: The Amarna Letters,http://oracc.museum.upenn.edu/aemw/amarna,aemw/amarna,Akkadian,no,no,14/05/2025
9,aemw/ugarit — AEMW Ugarit Corpus,https://oracc.museum.upenn.edu/aemw/ugarit/corpus,aemw/ugarit,"Akkadian, Ugaritic",no,no,14/05/2025


In [8]:
# Search for projects by keyword (searches all columns)
SEARCH = "rinap"

# fillna('') handles missing values safely before joining
mask = projects.apply(
    lambda r: SEARCH.lower() in ' '.join(r.fillna('').astype(str)).lower(),
    axis=1,
)
matches = projects[mask]
print(f"Found {len(matches)} projects matching '{SEARCH}':")
display(matches)

Found 9 projects matching 'rinap':


,Project,Link,Project_Name,Languages,Is_Umbrella_Project,Is_Text_Folder_Empty,Last_Check
154,RINAP: Royal Inscriptions of the Neo-Assyrian ...,http://oracc.museum.upenn.edu/rinap,rinap,Akkadian,yes,yes,14/05/2025
155,RINAP 1: Tiglath-pileser III and Shalmaneser V,http://oracc.museum.upenn.edu/rinap/rinap1,rinap/rinap1,Akkadian,no,no,14/05/2025
156,RINAP 2: Sargon II,http://oracc.museum.upenn.edu/rinap/rinap2,rinap/rinap2,Akkadian,no,no,14/05/2025
157,RINAP 3: Sennacherib,http://oracc.museum.upenn.edu/rinap/rinap3,rinap/rinap3,Akkadian,no,no,14/05/2025
158,RINAP 4: Esarhaddon,http://oracc.museum.upenn.edu/rinap/rinap4,rinap/rinap4,Akkadian,no,no,14/05/2025
159,RINAP 5: Ashurbanipal and Successors,http://oracc.museum.upenn.edu/rinap/rinap5,rinap/rinap5,Akkadian,no,no,14/05/2025
160,RINAP 5: Ashurbanipal and Successors,https://oracc.museum.upenn.edu/rinap/rinap5p1/,rinap/rinap5p1,Akkadian,no,no,14/05/2025
161,RINAP Scores,http://oracc.museum.upenn.edu/rinap/scores,rinap/scores,Akkadian,no,no,14/05/2025
162,RINAP Sources,http://oracc.museum.upenn.edu/rinap/sources,rinap/sources,Akkadian,no,no,14/05/2025


## 2. Provenance — Tablet Find Spots

ORACC tablets come with a raw `provenience` string (e.g. `"Nineveh"`, `"Assur"`).
We pre-built a mapping from these raw strings to:
- A **normalized city name**
- A **[Pleiades](https://pleiades.stoa.org/) ID** and coordinates (lat/lon)

This was built by:
1. Collecting all unique provenience strings from ORACC
2. Matching them to Pleiades gazeteer entries via API
3. Hand-verifying ambiguous cases

By default `get_provenance()` returns **only rows with a confirmed Pleiades ID**.
This is also what the pipeline uses to attach coordinates to parsed tablets.

In [9]:
# By default: only rows with a confirmed numeric Pleiades ID
prov = reference_data.get_provenance()
print(f"✅ {len(prov)} provenance mappings with confirmed Pleiades IDs")
display(prov.head(20))

✅ 262 provenance mappings with confirmed Pleiades IDs


,raw_provenience,normalized_city,pleiades_id,pleiades_title,lat,lon
0,Acharneh (Tunip),Tunip,668200,Asharne,35.2833333,36.4
1,Adab,Adab,787747618,Adab,31.9509002901,45.62333875145
2,Akhetaten (Mod. Amarna),el-Amarna,149576487,Amarna,27.644099597341675,30.89818203754354
3,Akhetaten (mod. el-Amarna),el-Amarna,149576487,Amarna,27.644099597341675,30.89818203754354
4,Akko,Akko,678010,Ake/Ptolemais,32.92128675,35.079835
5,Alalakh,Alalakh,309866128,Alalaḫ,36.23807544872566,36.38383192027712
6,Amarna,el-Amarna,149576487,Amarna,27.644099597341675,30.89818203754354
7,Ana,Ana,893936,Anatho,34.4678,41.9794
8,Antakya,Antakya,658381,Antiochia/Theoupolis,36.2225515,36.183214
9,Aqar Quf (Dur-Kurigalzu),Dur-Kurigalzu,893988,Dur-(Kuri)galzu,33.35607304048341,44.197416520972745


In [10]:
# To see ALL mappings (including uncertain/unmatched):
from oracc_parser.utils.paths import get_provenience
prov_all = get_provenience(pleiades_only=False)
print(f"🗺️  Total provenance mappings: {len(prov_all)}")
print(f"   With Pleiades ID:           {len(prov)} ({len(prov)/len(prov_all):.0%})")
print(f"   Without Pleiades ID:        {len(prov_all) - len(prov)} (uncertain / unidentified)")

🗺️  Total provenance mappings: 336
   With Pleiades ID:           262 (78%)
   Without Pleiades ID:        74 (uncertain / unidentified)


### 2.2 Historical Periods → Year Ranges

ORACC text metadata includes period names like `"Neo-Assyrian"`, `"Old Babylonian"`, etc.
We pre-saved a mapping from these period names to approximate year ranges.
The pipeline uses this to add `start_year` / `end_year` to every parsed tablet.

In [11]:
periods = reference_data.get_period_mapping()
print(f"{len(periods)} historical periods")
display(periods)

25 historical periods


,period_name,start_year,end_year
0,Old Akkadian,-2340.0,-2200.0
1,Ur III,-2100.0,-2000.0
2,Early Old Babylonian,-2000.0,-1900.0
3,Old Assyrian,-1950.0,-1850.0
4,Old Babylonian,-1900.0,-1600.0
5,Standard Babylonian,-1000.0,-1.0
6,Mitanni,-1550.0,-1260.0
7,Middle Hittite,-1500.0,-1100.0
8,Middle Babylonian,-1400.0,-1100.0
9,Middle Assyrian,-1400.0,-1000.0


### 2.3 Cuneiform Sign List (8,900+ readings)

This table maps cuneiform sign names to their Unicode code points.
The parser uses it to render transliterated text as actual cuneiform characters (𒀸, 𒈾, etc.).
It was compiled from the ORACC sign list.

In [14]:
signs = reference_data.get_sign_list()
print(f"{len(signs)} sign readings")
display(signs.head(20))

8902 sign readings


,reading,cuneiform
0,ʾu₄,𒀀
1,a,𒀀
2,aia₂,𒀀
3,aya₂,𒀀
4,barₓ,𒌋
5,buniŋₓ,𒆹
6,burₓ,𒋲
7,dur₅,𒀀
8,duru₅,𒀀
9,e₄,𒀀


In [15]:
# Search for a sign reading
SIGN_QUERY = "lugal"  # Try: "an", "dingir", "sar", "gal", "ki"

mask = signs.apply(
    lambda r: SIGN_QUERY.lower() in ' '.join(r.fillna('').astype(str)).lower(),
    axis=1,
)
matches = signs[mask]
print(f"Found {len(matches)} readings for '{SIGN_QUERY}':")
display(matches)

Found 2 readings for 'lugal':


,reading,cuneiform
4885,lugal,𒈗
4886,lugala,𒈗


### 2.4 POS Tags & Language Codes

**POS tags** — ORACC uses a custom tagset (e.g. `PN` = Personal Name, `GN` = Geographical Name).
We presaved a table linking each tag to its human-readable meaning, as well as a `Mask as` column for masking proper nouns (see notebook 3).

In [16]:
pos = reference_data.get_pos_tags()
print(f"There are {len(pos)} POS tags in total. Here are the major ones used for masking:")
major_masks = pos[pos['To mask'] == 'yes'][['POS-tag', 'Meaning', 'Mask as']].drop_duplicates()
for _, row in major_masks.head(10).iterrows():
    print(f"  '{row['POS-tag']}' ({row['Meaning']}) -> masked as {row['Mask as']}")
print("  ...")

There are 49 POS tags in total. Here are the major ones used for masking:
  'n' (Number) -> masked as NUM
  'PN' (Personal Name) -> masked as PN
  'DN' (Divine Name) -> masked as DN
  'SN' (State Name (city name)) -> masked as GN
  'CN' (Constellation Name) -> masked as CN
  'GN' (Geographical Name) -> masked as GN
  'RN' (Royal Name) -> masked as RN
  'MN' (Month Name) -> masked as MN
  'NU' (Number) -> masked as NUM
  'LN' (Lineage Name (ancestral name)) -> masked as PN
  ...


In [17]:
langs = reference_data.get_languages()
print(f"There are {len(langs)} language variants. The base languages are:")
base_langs = langs['language_name'].dropna().unique()
for l in sorted(base_langs):
    print(f"  - {l}")

There are 35 language variants. The base languages are:
  - Akkadian
  - Aramaic
  - Canaanite
  - Egyptian
  - Elamite
  - Hittite
  - Hurrian
  - Persian
  - Proto-Elamite
  - Proto-cuneiform
  - Sumerian
  - Ugaritic
  - Unclear


### 2.5 State/Region Supergroups

Each ORACC project is mapped to a broad geopolitical supergroup (e.g. `"Neo-Assyrian Empire"`, `"Old Babylonian Kingdoms"`, `"Western Periphery"`). This is a hand-curated classification created for this package that groups projects by the dominant political entity relevant to their texts.

The mapping uses two match strategies:
- **`exact`** — the project slug must match the `project` column exactly (e.g. `babcity` → `Mix`)
- **`prefix`** — all sub-projects whose slug starts with the given prefix inherit the same supergroup (e.g. `atae-*` → `Neo-Assyrian Empire`)

The `state_supergroup` field appears in the output of `get_metadata_table()` and the full flat table.

In [18]:
state_map = reference_data.get_state_supergroup_mapping()
print(f"{len(state_map)} project mappings covering {state_map['state_supergroup'].nunique()} supergroups")
print(f"\nSupergroups:")
for sg, count in state_map['state_supergroup'].value_counts().items():
    print(f"  {sg:40s}  ({count} {'prefix' if count > 1 else 'entry'})")
display(state_map)

56 project mappings covering 12 supergroups

Supergroups:
  Mix                                       (24 prefix)
  Old Babylonian Kingdoms                   (6 prefix)
  Neo-Assyrian Empire                       (5 prefix)
  Babylonia Transition Period 2nd to 1st mill  (5 prefix)
  Seleucid Empire                           (4 prefix)
  Neo-Babylonian Empire                     (4 prefix)
  Achemenid Empire                          (2 prefix)
  Western Periphery                         (2 prefix)
  Seleucid Empire; Parthian Empire          (1 entry)
  Suhu                                      (1 entry)
  Middle Assyrian State                     (1 entry)
  First Sealand Dynasty                     (1 entry)


,project,match_type,state_supergroup
0,adsd-adart1,exact,Mix
1,adsd-adart2,exact,Seleucid Empire
2,adsd-adart3,exact,Seleucid Empire; Parthian Empire
3,adsd-adart5,exact,Mix
4,adsd-adart6,exact,Mix
5,aemw-alalakh-idrimi,exact,Western Periphery
6,aemw-amarna,exact,Mix
7,aemw-ugarit,exact,Western Periphery
8,akklove,exact,Old Babylonian Kingdoms
9,ario,exact,Achemenid Empire


## What's next?

- **Notebook 01 — Quickstart:** Parse a project, explore individual tablets
- **Notebook 03 — Configure & Export:** Use `RunConfig` to mask POS tags, control output format, batch-export